## **03. Transfer Learning with DistilBERT**

**Objective**: Fine-tune a pre-trained Transformer model (DistilBERT) for maximum spam detection performance.

**Why DistilBERT?**
- Pre-trained on billions of words => deep language understanding
- 40% smaller & 60% faster than BERT, with 97% of its performance
- Perfect for small datasets like ours (5,572 samples)
- State-of-the-art on text classification tasks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, warnings
warnings.filterwarnings('ignore')

import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (DistilBertTokenizer, DistilBertForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score)
from sklearn.utils.class_weight import compute_class_weight
from dotenv import load_dotenv

load_dotenv()

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

spam_data_path = '../data/spam.csv'
outputs_model_path = '../outputs/models'

PyTorch: 2.12.0+cu130
Device: cpu


### **1. Data Loading**

In [4]:
df = pd.read_csv(spam_data_path, encoding='latin-1')
df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
df['label_encoded'] = (df['label'] == 'spam').astype(int)

X = df['text'].to_numpy()
y = df['label_encoded'].to_numpy()

# Stratified split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Spam ratios => Train: {y_train.mean():.3f} | Val: {y_val.mean():.3f} | Test: {y_test.mean():.3f}")

Train: 3900 | Val: 836 | Test: 836
Spam ratios => Train: 0.134 | Val: 0.134 | Test: 0.134


### **2. DistilBERT Tokenization**

Unlike our previous models, DistilBERT uses its own tokenizer (WordPiece) — no manual preprocessing needed! The tokenizer handles lowercasing, subword splitting, and special tokens ([CLS], [SEP]).

In [5]:
# Load DistilBERT tokenizer
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

# Determine optimal max length
lengths = [len(tokenizer.encode(text)) for text in X_train]
print(f"Token length stats:")
print(f"  Mean: {np.mean(lengths):.0f} | Median: {np.median(lengths):.0f}")
print(f"  95th percentile: {np.percentile(lengths, 95):.0f}")
print(f"  Max: {np.max(lengths)}")

MAX_LENGTH = min(int(np.percentile(lengths, 99)) + 2, 128)  # Cap at 128 for efficiency
print(f"\nUsing MAX_LENGTH = {MAX_LENGTH}")

# Show tokenization example
example = "CONGRATULATIONS! You won a free iPhone! Call 0800123456 NOW"
tokens = tokenizer.tokenize(example)
print(f"\nExample tokenization:")
print(f"Original: {example}")
print(f"Tokens: {tokens}")
print(f"IDs: {tokenizer.encode(example)}")

Token length stats:
  Mean: 25 | Median: 20
  95th percentile: 56
  Max: 238

Using MAX_LENGTH = 80

Example tokenization:
Original: CONGRATULATIONS! You won a free iPhone! Call 0800123456 NOW
Tokens: ['congratulations', '!', 'you', 'won', 'a', 'free', 'iphone', '!', 'call', '08', '##00', '##12', '##34', '##56', 'now']
IDs: [101, 23156, 999, 2017, 2180, 1037, 2489, 18059, 999, 2655, 5511, 8889, 12521, 22022, 26976, 2085, 102]


### **3. PyTorch Dataset & DataLoaders**

In [6]:
class SpamDataset(Dataset):
    """Custom PyTorch Dataset for spam classification."""
    
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
BATCH_SIZE = 32

train_dataset = SpamDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset = SpamDataset(X_val, y_val, tokenizer, MAX_LENGTH)
test_dataset = SpamDataset(X_test, y_test, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Batches Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Batches Train: 122 | Val: 27 | Test: 27


#### **Load Pre-trained DistilBERT**

We load `distilbert-base-uncased` with a classification head on top. The model has ~66M parameters, but we only fine-tune the last layers.

In [7]:
# Load pre-trained model
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,        # Binary: ham=0, spam=1
    dropout=0.3,
    attention_dropout=0.3
)
model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4365.34it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 66,955,010
Trainable parameters: 66,955,010


#### **Training Configuration**

In [8]:
# Hyperparameters
EPOCHS = 4           # DistilBERT converges fast - 2-4 epochs is usually enough
LEARNING_RATE = 2e-5 # Standard for fine-tuning transformers
WARMUP_RATIO = 0.1

# Optimizer with weight decay (AdamW)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Learning rate scheduler with warmup
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# Class weights for loss
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Class weights: ham={class_weights[0]:.3f}, spam={class_weights[1]:.3f}")

Total training steps: 488
Warmup steps: 48
Class weights: ham=0.577, spam=3.728


### **Training Loop**

In [9]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            
            total_loss += loss.item()
            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # P(spam)
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_probs), np.array(all_labels)

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train Acc':>10} | {'Val Acc':>10} | {'Val F1':>8}")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss, val_acc, val_preds, val_probs, val_labels = eval_epoch(model, val_loader, criterion, device)
    val_f1 = f1_score(val_labels, val_preds)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    marker = "best" if val_f1 > best_val_f1 else ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        # Save best model
        torch.save(model.state_dict(), f'{outputs_model_path}/best_distilbert_spam.pth')
    
    print(f"{epoch+1:>5} | {train_loss:>10.4f} | {val_loss:>10.4f} | {train_acc:>10.4f} | {val_acc:>10.4f} | {val_f1:>8.4f}{marker}")

print(f"\nTraining complete! Best Val F1: {best_val_f1:.4f}")

Epoch | Train Loss |   Val Loss |  Train Acc |    Val Acc |   Val F1
